## Indexing is the process of preparing your documents so they can be searched efficiently during retrieval.

## Instead of searching through every document each time a user asks a question, RAG first builds an index (a searchable vector database). Later, it uses that index to quickly find the most relevant information.


# 02. RAG Indexing

Building the indexing pipeline by loading documents, splitting them into token-based chunks, generating embeddings, and storing them in a Chroma vector database for efficient retrieval.


In [1]:
%run ./01_basic_rag.ipynb

1.3.13



C:\Users\Acer\AppData\Local\Temp\ipykernel_7852\2796628868.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


The full form of HTML is HyperText Markup Language.


In [3]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [ ]:
import tiktoken  # OpenAI ko BPE tokenizer use garna


# Diyeko text ma kati ota tokens chan bhanera count garne function.
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""

    # Chosen encoding load garne (BPE tokenizer)
    encoding = tiktoken.get_encoding(encoding_name)

    # Text lai tokens ma convert garera token ko length count garne
    num_tokens = len(encoding.encode(string))

    # Total token count return garne
    return num_tokens


# Question ma kati tokens chan bhanera print garne
num_tokens_from_string(question, "cl100k_base")

8

In [ ]:
from langchain_huggingface import (
    HuggingFaceEmbeddings,
)  # HuggingFace ko free embedding model use garna

# Text lai vector (embeddings) ma convert garne model load gareko.
embd = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# User ko question lai embedding vector ma convert gareko.
query_result = embd.embed_query(question)

# Document lai embedding vector ma convert gareko.
document_result = embd.embed_query(document)

# Question embedding ko dimension (vector size) print garne.
print(len(query_result))

# Document embedding ko dimension (vector size) print garne.
print(len(document_result))

384
384


In [ ]:
import numpy as np  # Mathematical operations ra vector calculations garna


# Dui ota vectors kati similar chan bhanera cosine similarity calculate garne function.
def cosine_similarity(vec1, vec2):

    # Dui vectors ko dot product nikalne.
    dot_product = np.dot(vec1, vec2)

    # Pahilo vector ko magnitude (length) nikalne.
    norm_vec1 = np.linalg.norm(vec1)

    # Dosro vector ko magnitude (length) nikalne.
    norm_vec2 = np.linalg.norm(vec2)

    # Cosine similarity calculate garera return garne.
    return dot_product / (norm_vec1 * norm_vec2)


# Question ra document ko embeddings bich ko similarity calculate garne.
similarity = cosine_similarity(query_result, document_result)

# Similarity score print garne.
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.7378822383225558


In [ ]:
import bs4  # HTML content parse garna BeautifulSoup use garne.
from langchain_community.document_loaders import (
    WebBaseLoader,
)  # Website bata documents load garna.

# Website bata data load garne loader banayeko.
loader = WebBaseLoader(
    # Data load garne website ko URL.
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    # HTML ko specific content matra extract garna.
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            # Yi HTML classes bhitra ko content matra load garne.
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

# Website bata documents load gareko.
blog_docs = loader.load()

In [ ]:
from langchain_text_splitters import TokenTextSplitter

# TokenTextSplitter use gareko, character ko satta token ko basis ma text split garna
# BPE (Byte Pair Encoding) tokenizer use huncha, jasle LLM le dekhne actual token anusar chunk banaucha
# chunk_size = 256 tokens, chunk_overlap = 50 tokens rakhiyeko cha context preserve garna

text_splitter = TokenTextSplitter(
    chunk_size=256,
    chunk_overlap=50,
)

# Loaded document lai token-based chunks ma split garne
splits = text_splitter.split_documents(blog_docs)

In [ ]:
from langchain_community.vectorstores import Chroma

# Chroma eutaa vector database ho, jasma document ko embeddings store garna sakincha
# split bhayeka documents ra embeddings use garera vector store create gareko

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
)

# Retriever banayeko, jasle query ko sabai bhanda similar chunks vector database bata search garera lyaucha
retriever = vectorstore.as_retriever()